# Chapter 36 — Model Access Without the Coding Agent

**Companion to *Pi Agents*** · [`Pi Agents` chapter 36](https://programmer.ie/books/pi/36-chapter/)

| | |
|---|---|
| Chapter | `36-chapter.md` · weight 36 · `/books/pi/36-chapter/` |
| Notebook | `notebooks/pi/36-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Can one `models.complete()` become a typed value, and what happens when the
arguments break the schema?

The chapter's point: `pi-ai` is a library. One `complete()` call can end in a
**typed value** when the model submits valid data, a **rejection** when the
arguments break the schema, an **error** when the model answers with prose, or
an **exception** when the request itself fails. The caller must distinguish all
four.


## What this notebook establishes

- `models.complete()` can return a **typed value** — the model's `submit_assessment` tool call is decoded and validated against the schema.
- Arguments that **break the schema are rejected**, not passed on.
- **Prose instead of a tool call** is an error the caller can see.
- A **failed request** is an exception with the provider's message, not a silent value, and one request costs exactly one call.

## What this notebook does **not** establish

- **Real-model behaviour is not measured** (`ch36-lim2`) — everything runs through the scripted provider.
- **The `builtinModels` / `classify` snippets** are documented API, not run.
- **One request count is exact for the scripted control flow only** (`ch36-lim2`).


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — the chapter's `assess` against the scripted provider (`createModels` + `fauxProvider`).
* **Evidence scope:** `in_process_runtime`. Schema validation and error surfaces are Pi's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(36))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/36-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: one call, a typed value, exactly one request

In [3]:
ACCESS = pinb.driver_source("ch36-model-access.ts")
acc = pinb.driver(ACCESS, label="ch36-model-access", timeout=180)
v = acc["valid"]
print(pinb.md_table(
    ["check", "value"],
    [["verdict", v["verdict"]],
     ["request count", v["callCount"]]],
))

| check | value |
|---|---|
| verdict | supported |
| request count | 1 |


In [4]:
pinb.show_checks([
    pinb.check("a valid submission comes back as a typed value",
               v["verdict"] == "supported", "supported"),
    pinb.check("it cost exactly one request", v["callCount"] == 1, "callCount 1"),
])

== Assertions ==
  assertion  observed
----  ----------------------------------------------  -----------
PASS  a valid submission comes back as a typed value  supported
PASS  it cost exactly one request  callCount 1

2/2 assertions held.


## Three failure shapes

In [5]:
sb, pr, fr = acc["schemaBreaking"], acc["prose"], acc["failedRequest"]
print(pinb.md_table(
    ["case", "rejected?", "detail"],
    [["schema-breaking arguments", "yes" if sb["rejected"] else "no", sb["error"]],
     ["prose instead of a tool call", "yes" if pr["rejected"] else "no", pr["error"]],
     ["failed request", "yes" if fr["rejected"] else "no", "provider message surfaced" if fr["hasProviderMessage"] else "-"]],
))

| case | rejected? | detail |
|---|---|---|
| schema-breaking arguments | yes | Validation failed for tool "submit_assessment":
  - verdict: must be equal to co |
| prose instead of a tool call | yes | the model answered in prose instead of calling submit_assessment |
| failed request | yes | provider message surfaced |


In [6]:
pinb.show_checks([
    pinb.check("schema-breaking arguments are rejected, not passed on",
               sb["rejected"], "rejected"),
    pinb.check("prose instead of a tool call is an error",
               pr["rejected"], "error"),
    pinb.check("a failed request is an exception with the provider message",
               fr["rejected"] and fr["hasProviderMessage"], "429 message surfaced"),
    pinb.check("the failed request still cost exactly one call",
               fr["callCount"] == 1, "callCount 1"),
])

== Assertions ==
  assertion  observed
----  ----------------------------------------------------------  --------------------
PASS  schema-breaking arguments are rejected, not passed on  rejected
PASS  prose instead of a tool call is an error  error
PASS  a failed request is an exception with the provider message  429 message surfaced
PASS  the failed request still cost exactly one call  callCount 1

4/4 assertions held.


## The chapter's own tests

All four model-access tests.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [7]:
PATTERNS = 'ch36-model-access/assess.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch36-model-access/assess.test.ts', show="schema")

$ node --test ch36-model-access/assess.test.ts
  PASS  4 passed, 0 failed, 0 skipped  in 614 ms

  tests matching 'schema':
    ok   arguments that break the schema are rejected, not passed on


## Your turn: predict, then run

**Predict first.** The schema requires `confidence` between 0 and 1. What does the
call return if the model sends `confidence: 7` — and what would it do if you
changed the schema to allow any number?

**Then change one input.** `assess()` takes `citations: string[]`. What happens if
the model's tool call omits `citations` entirely — is that a schema error or a
default?

**Predict the boundary.** A failed request is an exception. What kind of error is
a *rate limit* — a provider failure with the message, or something else?

In [8]:
print("Predictions:")
print("  1. confidence: 7 is rejected by the schema (the check is on the type).")
print("  2. Omitting citations: schema error if required, default if optional.")
print("  3. A rate limit surfaces as a provider failure exception with its message.")
print()
print("Observed above:")
print(f"  valid: verdict={v['verdict']}, callCount={v['callCount']}")
print(f"  schema-breaking rejected: {sb['rejected']}")
print(f"  prose rejected: {pr['rejected']}")
print(f"  failed request: rejected={fr['rejected']}, has provider message={fr['hasProviderMessage']}")
print()
assert v["verdict"] == "supported" and v["callCount"] == 1
assert sb["rejected"] and pr["rejected"] and fr["rejected"] and fr["hasProviderMessage"]
print("held: one call becomes a typed value; schema breaks are rejected; prose and failures are errors")

Predictions:
  1. confidence: 7 is rejected by the schema (the check is on the type).
  2. Omitting citations: schema error if required, default if optional.
  3. A rate limit surfaces as a provider failure exception with its message.

Observed above:
  valid: verdict=supported, callCount=1
  schema-breaking rejected: True
  prose rejected: True
  failed request: rejected=True, has provider message=True

held: one call becomes a typed value; schema breaks are rejected; prose and failures are errors


## Interpretation

`pi-ai` is a library with four outcomes, and the chapter shows the boundaries:

| Outcome | When | What the caller sees |
|---|---|---|
| **Typed value** | Valid tool call matching the schema | The decoded submission, one request |
| **Rejection** | Arguments break the schema | An error naming the problem |
| **Error** | Prose instead of a tool call | An error meaning "you asked for a tool call" |
| **Exception** | The request itself fails | The provider's message, one request |

Practical rules:
1. **A failed request is a value, not silence** — check `stopReason` and `errorMessage` first.
2. **The schema is the contract** — codify the tool's output type and decode it.
3. **Count requests** — a valid call costs one; a repair costs two.
4. **Distinguish the four outcomes** — a schema break is not a provider failure.

## Sources, execution mode and limitations

**Execution mode:** **Run** — the chapter's `assess` against the scripted provider (`createModels` + `fauxProvider`).

**Evidence scope:** `in_process_runtime`. Schema validation and error surfaces are Pi's.

### What was read or run

- **Ran** `drivers/ch36-model-access.ts`: the chapter's `assess` four ways (valid, schema-breaking, prose, failed request).
- **Ran** `ch36-model-access/assess.test.ts` (4 tests).
- **Read** `examples/evidence.json`, chapter 36 rows (4 claims).

### Limitations

- **Real-model behaviour is not measured** (`ch36-lim2`) — everything runs through the scripted provider.
- **The `builtinModels` / `classify` snippets** are documented API, not run.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
